# GiantMed: medication recommendation with SafeDrug and an LLM

This walkthrough explains the GiantMed pipeline implemented in PyHealth. It combines a trained medication model with similar visits, drug interaction evidence and an LLM that revises scores near a decision boundary.

**Audience:** readers familiar with Python and basic machine learning. **You will:** inspect a small synthetic example, understand the input token budget, and read recorded results from the MIMIC-III and MIMIC-IV demos.

The default notebook is offline. It does **not** call an LLM service or train on patient data. The toy example is fabricated; the comparisons are aggregate summaries recorded from five-visit demo runs with an earlier prompt renderer. They are not recomputed by the notebook and cannot be reproduced from this repository alone: the mapping files, expanded ingredient cache, trained backbone and, for Qwen, provider access are separate inputs. Neither demo is large enough to establish clinical utility or reproduce the paper. [Paper](https://doi.org/10.1145/3770854.3780297) · [Reference code](https://github.com/lvhangkenn/GiantMed)

## 1. The idea

1. SafeDrug predicts one probability per medication.
2. GiantMed identifies scores near the recommendation cutoff (the released refinement script uses 0.3 ≤ score < 0.7).
3. It prepares current-visit context, similar **training** visits and interactions among boundary candidates.
4. The LLM proposes revised boundary scores; unmodified scores retain the backbone prediction.

The LLM receives possible interactions as evidence. They are **not** enforced as a hard safety constraint.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Find the checkout when launched either at the repository root or in examples.
working = Path.cwd().resolve()
repo = next((p for p in (working, *working.parents) if (p / "pyhealth").is_dir()), None)
if repo is None:
    raise RuntimeError("Run this notebook from inside a PyHealth checkout.")
import contextlib, io, warnings
with warnings.catch_warnings(), contextlib.redirect_stderr(io.StringIO()):
    warnings.simplefilter("ignore")
    from pyhealth.models import GiantMed
print("PyHealth checkout:", repo.name)

## 2. A small synthetic walkthrough

These codes, visits, interactions and LLM answers are invented. A real model generates its answer; here a deterministic stand-in makes the example easy to run without network access.

In [ ]:
codes = ["A", "B", "C", "D"]
probabilities = np.array([0.91, 0.48, 0.53, 0.12])
ddi = np.zeros((4, 4), dtype=int)
ddi[1, 2] = ddi[2, 1] = 1
training = [{"patient_id": "train-1", "visit_id": "v1",
             "conditions": [["diagnosis-1"]], "procedures": [["procedure-1"]],
             "drugs": ["B"]}]
query = {"patient_id": "demo", "visit_id": "v2",
         "conditions": [["diagnosis-1"]], "procedures": [["procedure-1"]],
         "drugs_hist": [[]]}

def toy_llm(prompt):
    # Fabricated response, not clinical advice or a real model output.
    return '{"B": 0.78, "C": 0.25}'

refiner = GiantMed(codes, ddi, toy_llm, training)
result = refiner.refine_with_details(query, probabilities)
pd.DataFrame({"Drug": codes, "SafeDrug score": probabilities,
              "Refined score": result["probabilities"],
              "Boundary candidate": [c in result["candidates"] for c in codes]})

Only B and C fall in the boundary. In this toy response, B crosses the 0.5 cutoff upward and C crosses it downward; A and D are unchanged. The interaction information is included in the prompt, but a real LLM could still return an interacting combination.

In [ ]:
print("Candidates:", result["candidates"])
print("Accepted:", result["updated"])
print("Response status:", result["status"])
print("Interaction pair present:", '["B", "C"]' in result["prompt"])

## 3. Running the real MIMIC-III demo

The companion script loads the bundled MIMIC-III demo, splits patients, trains SafeDrug for one epoch, builds the retrieval index from training patients, and evaluates held-out visits. The output below **prints a command**; it does not execute it.

The example requires `--ndc-rxcui` and `--ndc-atc` for reference medication labels. Add `--medi` and `--ingredient-cache` to enable MEDI. The [README](giantmed_README.md) explains the required files, licensing and token-budget behavior. See the script for all options. A run using OpenRouter sends clinical and retrieved-visit information to the service and incurs API charges.

In [ ]:
print("Offline smoke test:")
print("python examples/drug_recommendation/giantmed_mimic3.py --ndc-rxcui PATH --ndc-atc PATH --demo-refinement")
print("\nReal LLM (after setting OPENROUTER_API_KEY in your environment):")
print("python examples/drug_recommendation/giantmed_mimic3.py "
      "--openrouter-model qwen/qwen3-14b --prompt-tokenizer Qwen/Qwen3-14B "
      "--medi PATH --ndc-rxcui PATH --ndc-atc PATH --ingredient-cache PATH")
print("\nThe printed commands are examples; no model call is made here.")

## 4. MIMIC-III: reference medication mapping

The example now uses the reference NDC → RxCUI → first ATC3 mapping for both task labels and MEDI evidence. The problematic NDC has no entry in this chain and is excluded; no substitute label is guessed. This avoids the known over-broad entry but does **not** repair PyHealth’s shared mapping resource. Of 8,920 usable prescription rows in the entire demo, 6,789 mapped; the rest were excluded.

The following one-epoch run covers five held-out visits. The weak one-epoch backbone, held-out label vocabulary exposure, small cohort and sampled model output make this a pipeline smoke test only. The expanded local RxNav ingredient cache was used; it is not bundled with PyHealth.

MEDI evaluates history rescue first; a rescued medication does not demonstrate ingredient-cache coverage.

In [ ]:
# Recorded aggregate results; no prompts, responses or patient identifiers.
reference_run = {'visits': 5, 'baseline': {'jaccard_samples': 0.25900159841467174, 'f1_samples': 0.40822082608196825, 'pr_auc_samples': 0.49918605283314055, 'ddi_rate': 0.04011194029850746, 'mean_medications': 21.25}, 'giantmed': {'jaccard_samples': 0.24820716320716318, 'f1_samples': 0.3936324206246823, 'pr_auc_samples': 0.3836284914071797, 'ddi_rate': 0.09941520467836257, 'mean_medications': 29.166666666666664}, 'medi': {'indication_mismatch': 106, 'indication_match': 102, 'history_rescue': 83}, 'requests': 15, 'cost': 0.00548486, 'prompt_tokens': [2097, 3424], 'candidate_names': [394, 394], 'ddi_pairs': [1383, 1383], 'cross_batch_pairs': 817, 'similar_visits': [45, 45], 'history_visits': [12, 12], 'prescription_ndc_rows': [6789, 8920]}
rows = []
for title, key in [("SafeDrug", "baseline"), ("GiantMed, reference mapping", "giantmed")]:
    x = reference_run[key]
    rows.append({"Method": title, "Jaccard": x["jaccard_samples"], "F1": x["f1_samples"], "PR-AUC": x["pr_auc_samples"], "DDI rate": x["ddi_rate"], "Mean medications": x["mean_medications"]})
pd.DataFrame(rows).set_index("Method").round(4)

In [ ]:
print("Usable prescription rows mapped:", reference_run["prescription_ndc_rows"])
print("MEDI outcomes:", reference_run["medi"])
print("Candidate names:", reference_run["candidate_names"])
print("Boundary interaction pairs:", reference_run["ddi_pairs"])
print("Similar-visit inclusions:", reference_run["similar_visits"])
print("History inclusions:", reference_run["history_visits"])
print("Requests / provider input-token range:", reference_run["requests"], reference_run["prompt_tokens"])
print("Reported cost (USD):", round(reference_run["cost"], 6))

## 5. Training SafeDrug for ten epochs

The same five visits, reference medication mapping, split, Qwen3-14B model and MEDI resources were used with a freshly trained SafeDrug backbone. The run evaluated the **final tenth epoch**; no early stopping or validation-based checkpoint selection was enabled. The validation set has only two samples. These sampled results illustrate the pipeline and cannot establish clinical benefit.

With the ten-epoch SafeDrug backbone, GiantMed had higher Jaccard and F1 on these visits, but lower PR-AUC and a higher DDI rate. The backbone predictions also changed with longer training. The task's label vocabulary can include held-out labels and unmapped prescription rows are excluded.

In [ ]:
# Recorded aggregates only; no patient prompts, responses or identifiers.
epochs10 = {'epochs': 10, 'visits': 5, 'baseline': {'jaccard_samples': 0.31937115857655585, 'f1_samples': 0.48228005358223147, 'pr_auc_samples': 0.615668007217317, 'ddi_rate': 0.08528784648187633, 'mean_medications': 14.166666666666668}, 'giantmed': {'jaccard_samples': 0.335519033976257, 'f1_samples': 0.500042376104532, 'pr_auc_samples': 0.6108673733360401, 'ddi_rate': 0.10182767624020887, 'mean_medications': 18.083333333333336}, 'decision_changes': {'added_recorded': 9, 'added_unrecorded': 23, 'removed_recorded': 3, 'removed_unrecorded': 10}, 'previous_decision_changes': {'added_recorded': 14, 'added_unrecorded': 74, 'removed_recorded': 9, 'removed_unrecorded': 39}, 'previous_candidates': 394, 'current_candidates': 99, 'ddi_pairs': [95, 95], 'requests': 5, 'cost': 0.00099198, 'prompt_tokens': [1074, 1872], 'validation_pr_auc': [0.4438, 0.4568, 0.5193, 0.5132, 0.5494, 0.5485, 0.5722, 0.5794, 0.599, 0.5776]}
rows = []
for title, key in [("SafeDrug, 10 epochs", "baseline"), ("GiantMed, 10 epochs", "giantmed")]:
    x = epochs10[key]
    rows.append({"Method": title, "Jaccard": x["jaccard_samples"], "F1": x["f1_samples"], "PR-AUC": x["pr_auc_samples"], "DDI rate": x["ddi_rate"], "Mean medications": x["mean_medications"]})
pd.DataFrame(rows).set_index("Method").round(4)

The boundary set decreased from 394 to 99 candidates. Counts below are pooled visit–medication decisions, while Jaccard and F1 are averaged within patients and then across patients. “Recorded” means present in the medication labels, not necessarily clinically appropriate. The prior one-epoch response counts were obtained by replaying the saved responses locally.

In [ ]:
print("One-epoch changes:", epochs10["previous_decision_changes"])
print("Ten-epoch changes:", epochs10["decision_changes"])
print("Boundary candidates, one vs ten epochs:", epochs10["previous_candidates"], epochs10["current_candidates"])
print("Validation PR-AUC by epoch:", epochs10["validation_pr_auc"])
print("Requests, prompt-token range and cost:", epochs10["requests"], epochs10["prompt_tokens"], round(epochs10["cost"], 6))

## 6. Try it yourself

**Exercise:** In the synthetic example, change B's proposed score to `0.45`. Would B be recommended at the 0.5 threshold? The cell below gives the answer without making a network request.

In [ ]:
exercise_score = 0.45
print("B recommended:", exercise_score >= 0.5)

**Answer:** No. `0.45` is below the 0.5 cutoff.

**Common pitfall:** passing a DDI matrix or medication codes in a different order from the backbone's output can produce incorrect evidence. Keep the matrix, codes and prediction vector aligned. For a fuller experiment, compare the same patient split and preprocessing under multiple seeds and document all omitted evidence and provider token counts.

## 7. Try a prompt policy

The instruction template is a patient-independent, single-line string containing `{boundary}`. GiantMed still constructs evidence sections and the JSON response contract. Custom instructions can change behavior and require evaluation. The example below uses invented data and no network.

In [ ]:
custom = GiantMed(
    ["A"], [[0]], lambda prompt: '{"A": 0.8}', [],
    prompt_template="Assess {boundary} conservatively and consider interactions.",
)
result = custom.refine_with_details({"patient_id": "synthetic"}, [0.5])
print("Refined score:", result["probabilities"][0])
print("Template SHA-256:", custom.prompt_metadata()["template_sha256"])

## 8. MIMIC-IV demo: ICD-aware MEDI evidence

The open [MIMIC-IV demo v2.2](https://physionet.org/content/mimic-iv-demo/2.2/) contains both ICD-9-CM and ICD-10-CM diagnoses. MEDI records for the two systems are loaded separately: a `9_` diagnosis is matched only against ICD-9 indications and `10_` only against ICD-10. Dots and case are normalized. Unprefixed codes retain the MIMIC-III ICD-9 interpretation; no crosswalk is inferred. When enabled, history rescue is checked before indication matching.

The aggregate results below come from one SafeDrug training epoch and five held-out visits (patient split, seed 0). This is a local pipeline smoke test with a **deterministic fake responder that assigns 0.8 to every boundary medication**. Its scores and decision changes have no clinical meaning; they do not measure GiantMed with an LLM. No MIMIC-IV prompts were sent externally. The mapping and ingredient cache are incomplete, and the 100-patient demo does not reproduce the paper.

In [ ]:
# Aggregate output only; no patient identifiers, prompts or responses.
mimic4 = {
    "sample_split_train_val_test": [102, 21, 9],
    "patient_split_train_val_test": [30, 4, 4],
    "mapped_usable_prescription_rows": [9380, 15456],
    "evidence": {"ICD-9": {"indication_match": 13, "indication_mismatch": 25, "history_rescue": 32},
                 "ICD-10": {"indication_match": 19, "indication_mismatch": 30, "history_rescue": 37}},
    "accepted_boundary_scores": [241, 241],
    "baseline": {"Jaccard": 0.2740, "F1": 0.4294, "PR-AUC": 0.5093, "DDI rate": 0.0155},
    "fake_refinement": {"Jaccard": 0.2665, "F1": 0.4202, "PR-AUC": 0.2765, "DDI rate": 0.0423},
}
print("Train / validation / test samples:", mimic4["sample_split_train_val_test"])
print("Train / validation / test patients:", mimic4["patient_split_train_val_test"])
print("Mapped / usable prescription rows:", mimic4["mapped_usable_prescription_rows"])
display(pd.DataFrame(mimic4["evidence"]).T)
display(pd.DataFrame({"SafeDrug": mimic4["baseline"], "Fake refinement": mimic4["fake_refinement"]}))

Counts above are retrieved-neighbour medication evidence checks, not unique patients. Jaccard, F1 and PR-AUC average visits within patients and then patients; DDI rate pools predicted medication pairs. Because the fake responder deliberately assigns 0.8 to every boundary drug, the table is useful for checking code paths only.

To repeat the run, download the demo and the separately supplied reference NDC, MEDI and ingredient files, then run:

```sh
python examples/drug_recommendation/giantmed_mimic4.py --root /path/to/mimic-iv-demo --ndc-rxcui /path/to/ndc2rxnorm_mapping.txt --ndc-atc /path/to/ndc2atc_level4.csv --medi /path/to/MEDI.csv --ingredient-cache /path/to/ingredients.json --demo-refinement --max-test-visits 5
```

Omit `--demo-refinement` for a no-op response, or configure a suitable model. A remote model receives clinical prompt data; use it only under appropriate data and provider terms.